# PlanCatalyst Dashboard Explorer

**Interactive prototype** of the Power BI dashboard — two modes, three sections:

| Mode | Section | What it does |
|------|---------|-------------|
| **Heatmap Mode** | §1 | Select one indicator → see all countries coloured by score |
| **Country Selector Mode** | §2 | Select a country → see all its indicators organised by sector |
| **Region Comparison** | §3 | Compare two continents on any indicator or sector |

---

### Audiences
- **PM (Thomas)**: Visual QA of scored data before uploading to Azure Blob.
- **Power BI developer**: Reference implementation — how to load, filter, group, and aggregate the data.
- **Client demos**: Walk through the notebook to preview dashboard views.

### Prerequisites
Run the full pipeline first so `data/interim/validated/` is populated:
```bash
python -m src.pipeline.main
```

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Make sure the project root is on the path so src.* imports resolve
_here = Path().resolve()
_root = _here if (_here / "src").is_dir() else _here.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import logging
import warnings
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)   # suppress INFO noise in notebook
%matplotlib inline

from src.plotting.dashboard_data import DashboardDataLoader
from src.plotting.dashboard_plotter import HeatmapModePlotter, CountryModePlotter
from src.utils.country_names import REGION_NAMES

# ── Initialise data loader and plotters ──────────────────────────────────────
loader  = DashboardDataLoader()
heatmap = HeatmapModePlotter(loader)
country_mode = CountryModePlotter(loader)

print("✓ Data loader and plotters ready")
print(f"  Data dir: {loader.data_dir}")

In [ ]:
# ── Build shared widget option lists ─────────────────────────────────────────

# --- Indicator dropdown options (grouped by sector / subsector) ---
def _build_indicator_options():
    """Return widget options list grouped by sector > subsector."""
    hierarchy = loader.get_indicator_hierarchy()
    options = []
    for domain_id, domain in sorted(hierarchy.items()):
        for sector_id, sector in sorted(domain["sectors"].items()):
            options.append((f"── {sector_id}  {sector['name']} ──", None))
            for sub_id, sub in sorted(sector["subsectors"].items()):
                options.append((f"   [{sub['name']}]", None))
                for ind in sub["indicators"]:
                    label = f"     {ind['indicator_id']}  {ind['name'][:55]}"
                    options.append((label, ind["indicator_id"]))
    return options

INDICATOR_OPTIONS = _build_indicator_options()

# --- Country options: ("Afghanistan (AFG)", "AFG") ---
COUNTRY_OPTIONS = [
    (f"{name} ({code})", code)
    for code, name in loader.get_country_list()
]

# --- Sector options ---
SECTOR_OPTIONS = [
    (f"{sid}  {sname}", sid)
    for sid, sname in loader.get_sector_list()
]

# --- Year range (from domain scores) ---
_dom = loader.load_domain_scores()
YEAR_MIN = int(_dom["year"].min())
YEAR_MAX = int(_dom["year"].max())

# --- Region options ---
REGION_OPTIONS = [
    (f"{rname} ({rcode})", rcode)
    for rcode, rname in loader.get_region_list()
]

print(f"✓ {len(INDICATOR_OPTIONS)} indicator dropdown entries  "
      f"({sum(1 for _, v in INDICATOR_OPTIONS if v)} selectable)")
print(f"✓ {len(COUNTRY_OPTIONS)} countries available")
print(f"✓ Year range: {YEAR_MIN} – {YEAR_MAX}")

---
## §1  Heatmap Mode

**Power BI equivalent**: Select an indicator from the right sidebar → globe colours all countries 0–100.

Controls:
- **Indicator** — hierarchical dropdown (sector → subsector → indicator)
- **Year** — slider; leave at `All years` (0) for the full time heatmap
- **View** — all countries vs region averages
- **Distribution** — show score histogram instead of heatmap

In [ ]:
# ── §1 Widget definitions ─────────────────────────────────────────────────────
_first_real = next(v for _, v in INDICATOR_OPTIONS if v is not None)

hm_indicator = widgets.Dropdown(
    options=INDICATOR_OPTIONS,
    value=_first_real,
    description="Indicator:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="600px"),
)
hm_year = widgets.IntSlider(
    min=0,
    max=YEAR_MAX,
    value=0,
    step=1,
    description="Year (0=all):",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="500px"),
    readout_format="d",
)
hm_view = widgets.RadioButtons(
    options=[
        ("All countries", "countries"),
        ("Region averages", "regions"),
        ("Score distribution", "distribution"),
    ],
    value="countries",
    description="View:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="300px"),
)
hm_save = widgets.Checkbox(
    value=False,
    description="Save plot to data/plots/",
    layout=widgets.Layout(width="220px"),
)
hm_btn = widgets.Button(
    description="Generate plot",
    button_style="primary",
    layout=widgets.Layout(width="150px"),
)
hm_out = widgets.Output()

def _on_hm_click(_btn):
    with hm_out:
        clear_output(wait=True)
        ind_id = hm_indicator.value
        if ind_id is None:
            print("⚠ Please select a specific indicator (not a header row).")
            return
        year = int(hm_year.value) if hm_year.value > 0 else None
        view = hm_view.value
        save = hm_save.value
        try:
            if view == "distribution":
                fig = heatmap.plot_score_distribution(ind_id, year=year, save=save)
            elif view == "regions":
                fig = heatmap.plot_indicator_heatmap_by_region(ind_id, year=year, save=save)
            else:
                fig = heatmap.plot_indicator_heatmap(ind_id, year=year, save=save)
            plt.show()
            plt.close(fig)
        except Exception as exc:
            print(f"❌ {exc}")

hm_btn.on_click(_on_hm_click)

display(
    widgets.VBox([
        widgets.HBox([hm_indicator]),
        widgets.HBox([hm_year]),
        widgets.HBox([hm_view, hm_save]),
        hm_btn,
        hm_out,
    ])
)

In [ ]:
# ── §1b  Print the hierarchical indicator selector list ──────────────────────
# Shows exactly how the Power BI right-sidebar indicator list should be structured.
print(heatmap.generate_indicator_selector_list())

---
## §2  Country Selector Mode

**Power BI equivalent**: Click a country on the globe → right sidebar shows that country's indicators.

Views:
| View | Description |
|------|-------------|
| **Profile** | One panel per sector, all indicator scores |
| **Sector detail** | Drill into one sector — subsector + indicator bars |
| **Indicator trend** | Score over time for one indicator |
| **Country vs Region** | Country trend vs its continental average |
| **Compare countries** | Two countries on one indicator or sector |

In [ ]:
# ── §2 Widget definitions ─────────────────────────────────────────────────────
_default_country = next(
    (code for code, _ in loader.get_country_list() if code == "AFG"),
    COUNTRY_OPTIONS[0][1],
)

cm_country_a = widgets.Dropdown(
    options=COUNTRY_OPTIONS,
    value=_default_country,
    description="Country A:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)
cm_country_b = widgets.Dropdown(
    options=[("(none)", None)] + COUNTRY_OPTIONS,
    value=None,
    description="Country B:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)
cm_view = widgets.RadioButtons(
    options=[
        ("Profile  (all sectors)", "profile"),
        ("Sector detail", "sector"),
        ("Indicator trend", "trend"),
        ("Country vs Region avg", "vs_region"),
        ("Compare countries", "compare"),
    ],
    value="profile",
    description="View:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="300px"),
)
cm_sector = widgets.Dropdown(
    options=SECTOR_OPTIONS,
    value=SECTOR_OPTIONS[0][1],
    description="Sector:",
    style={"description_width": "55px"},
    layout=widgets.Layout(width="360px"),
)
_first_ind = next(v for _, v in INDICATOR_OPTIONS if v is not None)
cm_indicator = widgets.Dropdown(
    options=INDICATOR_OPTIONS,
    value=_first_ind,
    description="Indicator:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="540px"),
)
cm_year = widgets.IntSlider(
    min=0,
    max=YEAR_MAX,
    value=0,
    step=1,
    description="Year (0=latest):",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="500px"),
    readout_format="d",
)
cm_save = widgets.Checkbox(
    value=False,
    description="Save plot to data/plots/",
    layout=widgets.Layout(width="220px"),
)
cm_btn = widgets.Button(
    description="Generate plot",
    button_style="primary",
    layout=widgets.Layout(width="150px"),
)
cm_out = widgets.Output()

def _on_cm_click(_btn):
    with cm_out:
        clear_output(wait=True)
        view    = cm_view.value
        ctry_a  = cm_country_a.value
        ctry_b  = cm_country_b.value
        sector  = cm_sector.value
        ind_id  = cm_indicator.value
        year    = int(cm_year.value) if cm_year.value > 0 else None
        save    = cm_save.value

        try:
            if view == "profile":
                fig = country_mode.plot_country_profile(ctry_a, year=year, save=save)

            elif view == "sector":
                fig = country_mode.plot_country_sector_detail(
                    ctry_a, sector, year=year, save=save
                )

            elif view == "trend":
                if ind_id is None:
                    print("⚠ Select a specific indicator for trend view.")
                    return
                fig = country_mode.plot_country_trend(ctry_a, ind_id, save=save)

            elif view == "vs_region":
                if ind_id is None:
                    print("⚠ Select a specific indicator.")
                    return
                fig = country_mode.plot_country_vs_region(ctry_a, ind_id, save=save)

            elif view == "compare":
                if ctry_b is None:
                    print("⚠ Select Country B for comparison.")
                    return
                if ind_id is not None:
                    fig = country_mode.plot_compare_countries(
                        ctry_a, ctry_b, indicator_id=ind_id, save=save
                    )
                else:
                    fig = country_mode.plot_compare_countries(
                        ctry_a, ctry_b, sector_id=sector, year=year, save=save
                    )

            plt.show()
            plt.close(fig)

        except Exception as exc:
            print(f"❌ {exc}")

cm_btn.on_click(_on_cm_click)

# Helper: show/hide sector and indicator controls based on view
def _on_view_change(change):
    v = change["new"]
    needs_sector  = v in ("sector", "compare")
    needs_ind     = v in ("trend", "vs_region", "compare")
    needs_year    = v in ("profile", "sector", "compare")
    needs_country_b = v == "compare"
    cm_sector.layout.visibility    = "visible" if needs_sector    else "hidden"
    cm_indicator.layout.visibility = "visible" if needs_ind       else "hidden"
    cm_year.layout.visibility      = "visible" if needs_year      else "hidden"
    cm_country_b.layout.visibility = "visible" if needs_country_b else "hidden"

cm_view.observe(_on_view_change, names="value")
_on_view_change({"new": cm_view.value})  # apply initial state

display(
    widgets.VBox([
        widgets.HBox([cm_country_a, cm_country_b]),
        widgets.HBox([cm_view]),
        widgets.VBox([cm_sector, cm_indicator, cm_year]),
        widgets.HBox([cm_btn, cm_save]),
        cm_out,
    ])
)

---
## §3  Region Comparison

**Power BI equivalent**: Left sidebar groups countries by continent → compare two regions.

Compare two continents on:
- An **indicator** (time series of region averages)
- A **sector** (grouped bar chart of all sector indicators, using region averages)

In [ ]:
# ── §3 Widget definitions ─────────────────────────────────────────────────────
rc_region_a = widgets.Dropdown(
    options=REGION_OPTIONS,
    value="AF",
    description="Region A:",
    style={"description_width": "75px"},
    layout=widgets.Layout(width="300px"),
)
rc_region_b = widgets.Dropdown(
    options=REGION_OPTIONS,
    value="AS",
    description="Region B:",
    style={"description_width": "75px"},
    layout=widgets.Layout(width="300px"),
)
rc_mode = widgets.RadioButtons(
    options=[
        ("Indicator (time series)", "indicator"),
        ("Sector (bar chart)", "sector"),
    ],
    value="indicator",
    description="Compare on:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="300px"),
)
rc_indicator = widgets.Dropdown(
    options=INDICATOR_OPTIONS,
    value=_first_real,
    description="Indicator:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="540px"),
)
rc_sector = widgets.Dropdown(
    options=SECTOR_OPTIONS,
    value=SECTOR_OPTIONS[0][1],
    description="Sector:",
    style={"description_width": "55px"},
    layout=widgets.Layout(width="360px"),
)
rc_year = widgets.IntSlider(
    min=0,
    max=YEAR_MAX,
    value=0,
    step=1,
    description="Year (0=latest):",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="500px"),
    readout_format="d",
)
rc_save = widgets.Checkbox(
    value=False,
    description="Save plot",
    layout=widgets.Layout(width="120px"),
)
rc_btn = widgets.Button(
    description="Generate plot",
    button_style="primary",
    layout=widgets.Layout(width="150px"),
)
rc_out = widgets.Output()

def _on_rc_mode_change(change):
    use_ind  = change["new"] == "indicator"
    rc_indicator.layout.visibility = "visible" if use_ind  else "hidden"
    rc_sector.layout.visibility    = "hidden"  if use_ind  else "visible"
    rc_year.layout.visibility      = "hidden"  if use_ind  else "visible"

rc_mode.observe(_on_rc_mode_change, names="value")
_on_rc_mode_change({"new": rc_mode.value})

def _on_rc_click(_btn):
    with rc_out:
        clear_output(wait=True)
        r_a  = rc_region_a.value
        r_b  = rc_region_b.value
        mode = rc_mode.value
        save = rc_save.value
        year = int(rc_year.value) if rc_year.value > 0 else None

        if r_a == r_b:
            print("⚠ Select two different regions.")
            return

        try:
            if mode == "indicator":
                ind_id = rc_indicator.value
                if ind_id is None:
                    print("⚠ Select a specific indicator.")
                    return
                fig = country_mode.plot_compare_regions(
                    r_a, r_b, indicator_id=ind_id, save=save
                )
            else:
                sector = rc_sector.value
                fig = country_mode.plot_compare_regions(
                    r_a, r_b, sector_id=sector, year=year, save=save
                )
            plt.show()
            plt.close(fig)
        except Exception as exc:
            print(f"❌ {exc}")

rc_btn.on_click(_on_rc_click)

display(
    widgets.VBox([
        widgets.HBox([rc_region_a, rc_region_b]),
        rc_mode,
        widgets.VBox([rc_indicator, rc_sector, rc_year]),
        widgets.HBox([rc_btn, rc_save]),
        rc_out,
    ])
)

---
## §4  Quick Data Inspection

Convenience cells for directly inspecting the structured output — useful for QA and as a Power BI developer reference.

In [ ]:
# ── Domain scores — top-level aggregated scores per country per year ──────────
domains = loader.load_domain_scores()
print(f"domain_scores: {domains.shape}  columns: {list(domains.columns)}")
domains.head(10)

In [ ]:
# ── Sector scores ─────────────────────────────────────────────────────────────
sectors = loader.load_sector_scores()
print(f"sector_scores: {sectors.shape}  columns: {list(sectors.columns)}")
sectors.head(10)

In [ ]:
# ── Subsector scores ──────────────────────────────────────────────────────────
subsectors = loader.load_subsector_scores()
print(f"subsector_scores: {subsectors.shape}  columns: {list(subsectors.columns)}")
subsectors.head(10)

In [ ]:
# ── Indicator scores (full) ───────────────────────────────────────────────────
inds = loader.load_indicator_scores()
print(f"Indicator_Scores_Full: {inds.shape}  columns: {list(inds.columns)}")
print(f"Indicators present:    {sorted(inds['indicator'].unique())}")
inds.head(10)

In [ ]:
# ── Country list (for Power BI slicer reference) ──────────────────────────────
import pandas as pd
country_df = pd.DataFrame(loader.get_country_list(), columns=["country_code", "display_name"])
print(f"{len(country_df)} countries  (join key: country_code)")
country_df.head(20)

In [ ]:
# ── Indicator hierarchy summary ───────────────────────────────────────────────
# Each indicator shows its sector / subsector grouping
rows = []
h = loader.get_indicator_hierarchy()
for did, domain in sorted(h.items()):
    for sid, sector in sorted(domain["sectors"].items()):
        for ssid, sub in sorted(sector["subsectors"].items()):
            for ind in sub["indicators"]:
                rows.append({
                    "domain": domain["name"],
                    "sector_id": sid,
                    "sector": sector["name"],
                    "subsector_id": ssid,
                    "subsector": sub["name"],
                    "indicator_id": ind["indicator_id"],
                    "name": ind["name"],
                })
pd.DataFrame(rows)